# Notebook 3: Analysis Dataset Builder

Ingests the pre-materialized assets from Stage 1 and Stage 2 and compiles them into a unified **33-band multi-scale analysis dataset** exported to Google Drive as GeoTIFFs.

### Output GeoTIFF Structure (33 bands total):
1. **`frip`** — 1 band: Cross-sectional Spearman correlation at that scale
2. **`FRIP_2001` ... `FRIP_2023`** — 23 bands: Annual Spearman correlations at that scale
3. **`uoi`**, **`rh98`**, **`gedi_n`** — 3 GEDI bands (openness, height, footprint count) aggregated to that scale
4. **`elevation`**, **`slope`**, **`hnd`**, **`precip`**, **`clay`**, **`forest_fraction`** — 6 covariate bands aggregated to that scale

### Running downstream:
The resulting GeoTIFFs can be downloaded locally and loaded natively by R using `terra::rast()` to perform hypothesis testing (NB4) and model coefficient extraction!

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")

# GEE Asset root
ASSET_ROOT = 'projects/quantum-bonus-434714-t2/assets/DefaunationFromSpace'

# Study regions (must match NB1 & NB2)
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
BASINS = [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX)]

# Scales
SCALES = list(range(5000, 105000, 5000))  # 5km to 100km in 5km steps
YEARS = list(range(2001, 2024))

print("\u2713 Configurations loaded.")
print(f"  Basins: {[b[0] for b in BASINS]}")
print(f"  Scales: {len(SCALES)} scales ({SCALES[0]/1000:.0f}km to {SCALES[-1]/1000:.0f}km)")

In [ ]:
# =============================================================================
# BLOCK 2: DATASET ASSEMBLY LOGIC
# =============================================================================

def build_scale_stack(basin_name, scale):
    """Assembles all 33 bands at the specified scale and basin.
    
    1. Loads pre-computed Stage 2 FRIP (cross-sectional + annual) at the target scale.
    2. Loads GEDI masked and Base Stack assets at native MODIS scale (~463m),
       and aggregates them to the target scale.
    """
    # 1. Load pre-computed FRIP assets (cross-sectional + annual) at target scale
    frip_cross = ee.Image(f'{ASSET_ROOT}/FRIP/FRIP_{scale}_{basin_name}').rename('frip')
    frip_annual = ee.Image(f'{ASSET_ROOT}/FRIP/FRIP_Annual_{scale}_{basin_name}')
    
    # 2. Load GEDI masked asset and aggregate to target scale
    gedi_base = ee.Image(f'{ASSET_ROOT}/GEDI/GEDI_masked_{basin_name}')
    gedi_proj = gedi_base.projection()
    
    uoi_agg = gedi_base.select('GEDI_UOI').setDefaultProjection(gedi_proj).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).rename('uoi')
    
    rh98_agg = gedi_base.select('GEDI_rh98').setDefaultProjection(gedi_proj).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).rename('rh98')
    
    n_agg = gedi_base.select('GEDI_N').setDefaultProjection(gedi_proj).reduceResolution(
        reducer=ee.Reducer.sum(), maxPixels=65535
    ).rename('gedi_n')
    
    # 3. Load Base Stack covariates and aggregate to target scale
    base = ee.Image(f'{ASSET_ROOT}/BaseStack_{basin_name}')
    base_proj = base.projection()
    
    covariates = ['elevation', 'slope', 'hnd', 'precip', 'clay', 'forest_fraction']
    covs_agg = base.select(covariates).setDefaultProjection(base_proj).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    )
    
    # 4. Concatenate all 33 bands
    stack = ee.Image.cat([
        frip_cross,    # 1 band (cross Spearman r)
        frip_annual,   # 23 bands (annual Spearman r)
        uoi_agg,       # 1 band (mean openness)
        rh98_agg,      # 1 band (mean height)
        n_agg,         # 1 band (sum footprint counts)
        covs_agg       # 6 bands (mean covariates)
    ]).toFloat()
    
    return stack

print("\u2713 Dataset assembly logic loaded.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Running assembly unit tests (using Congo at 50km)...\n")
    passed = 0
    total = 3
    
    try:
        # Use Congo at 50km for testing
        stack = build_scale_stack('Congo', 50000)
        bands = stack.bandNames().getInfo()
        
        # Test 1: Band count (33 bands expected)
        assert len(bands) == 33, f"Expected 33 bands, got {len(bands)}: {bands}"
        passed += 1
        print("  [1/3] \u2713 Correct band count: 33 bands assembled")
        
        # Test 2: Required bands present
        required = ['frip', 'FRIP_2001', 'FRIP_2023', 'uoi', 'rh98', 'gedi_n',
                    'elevation', 'slope', 'hnd', 'precip', 'clay', 'forest_fraction']
        missing = [b for b in required if b not in bands]
        assert not missing, f"Missing required bands: {missing}"
        passed += 1
        print("  [2/3] \u2713 All required signals and covariates present")
        
        # Test 3: Check projection is standard EPSG:4326
        # The base stack is now EPSG:4326, so our aggregated stack inherits it
        proj_info = stack.projection().getInfo()
        assert proj_info['crs'] == 'EPSG:4326', f"Expected EPSG:4326, got: {proj_info['crs']}"
        passed += 1
        print(f"  [3/3] \u2713 Geodetic reference standard: {proj_info['crs']}")
        
    except Exception as e:
        print(f"  \u2717 Test failed: {e}")
        print("    Ensure Stage 1 and Stage 2 assets have completed first!")
        
    print(f"\n{'='*60}")
    if passed == total:
        print(f"  \u2713 ALL {passed}/{total} TESTS PASSED")
        print("  Ready to configure Drive exports.")
    else:
        print(f"  \u2717 {passed}/{total} passed. Fix failures before proceeding.")
    print(f"{'='*60}")

# Run tests if GEE assets exist (will print warnings if assets aren't exported yet)
try:
    run_unit_tests()
except Exception as e:
    print(f"Skipping server run: {e}")

In [ ]:
# =============================================================================
# BLOCK 4: EXPORT TO DRIVE
# =============================================================================

def export_analysis_stacks(dry_run=True):
    """Launches exports for all 20 scales and both basins to Google Drive.
    
    Produces 40 GeoTIFFs total (20 scales x 2 basins).
    Saves to: 'DefaunationSynthesis/AnalysisStack/'
    """
    tasks = []
    
    for basin_name, basin_geom in BASINS:
        for scale in SCALES:
            stack = build_scale_stack(basin_name, scale)
            
            # Setup export task
            task = ee.batch.Export.image.toDrive(
                image=stack,
                description=f'analysis_stack_{scale}_{basin_name}',
                folder='DefaunationSynthesis/AnalysisStack',
                fileNamePrefix=f'analysis_stack_{scale}_{basin_name}',
                region=basin_geom,
                scale=scale,
                crs='EPSG:4326',
                maxPixels=1e13
            )
            tasks.append((task, f'analysis_stack_{scale}_{basin_name}'))
            
    print(f"\u2713 {len(tasks)} Drive export tasks configured:")
    print(f"  ({len(SCALES)} scales x {len(BASINS)} basins)")
    
    if dry_run:
        print("\nDRY RUN. Call export_analysis_stacks(dry_run=False) to launch.")
    else:
        for task, name in tasks:
            task.start()
            print(f"  \u2713 Started Drive export: {name}")
        print("\n\u2713 All 40 Drive exports started!")
        print("  Monitor at: https://code.earthengine.google.com/tasks")
        print("  Once done, sync your Google Drive to your local workspace's data/processed/ folder.")

export_analysis_stacks(dry_run=True)